## Value Counts & Unique

### Why does this matter?
Before you filter, group, or clean any categorical column, you need to know what values actually exist in it. How many unique departments are there? Which city appears most often? Are there typos like "Mumbaii" alongside "Mumbai"? Are there unexpected categories? These questions are answered in seconds with *value_counts()*, *unique()*, and *nunique()*. This is the first thing you run on any categorical column — before everything else.

### The three methods and what they each do

**unique()** — tells you what values exist. Returns the distinct values as an array.
**nunique() **— tells you how many distinct values exist. Returns a single integer.
**value_counts()** — tells you how many times each value appears. Returns a Series sorted by frequency. This is the most powerful of the three and the one you'll use most in real analysis.

### Why are these separate methods?
Because they answer three different questions. Sometimes you just need the count of unique values (for a quick cardinality check). Sometimes you need to see all unique values (to spot typos). Sometimes you need the full frequency distribution (for EDA and reporting). Having them separate keeps each operation clean and fast.

In [1]:
import pandas as pd
import numpy as np

data = {
    'Name':   ['Sumed','Priya','Rahul','Neha','Arjun',
               'Divya','Karan','Meera','Ravi','Pooja'],
    'Dept':   ['Data','HR','Data','Finance','Data',
               'HR','Finance','Data','data','HR'],  # 'data' is a typo!
    'City':   ['Mumbai','Delhi','Mumbai','Pune','Delhi',
               'Mumbai','Pune','Bangalore','Mumbai','Delhi'],
    'Salary': [60000,45000,75000,55000,80000,
               62000,71000,58000,67000,np.nan]
}
df = pd.DataFrame(data)

In [2]:
df

,Name,Dept,City,Salary
0,Sumed,Data,Mumbai,60000.0
1,Priya,HR,Delhi,45000.0
2,Rahul,Data,Mumbai,75000.0
3,Neha,Finance,Pune,55000.0
4,Arjun,Data,Delhi,80000.0
5,Divya,HR,Mumbai,62000.0
6,Karan,Finance,Pune,71000.0
7,Meera,Data,Bangalore,58000.0
8,Ravi,data,Mumbai,67000.0
9,Pooja,HR,Delhi,NaN


*Row 8 has 'data' (lowercase) instead of 'Data' — a typo. Row 9 has NaN salary. value_counts() will reveal both issues instantly.*

## Unique() : what values exist

In [4]:
# unique() → returns all distinct values as a NumPy array
# Order: first appearance order, NOT alphabetical
# Works on a Series (single column) only

df['Dept'].unique()
# → ['Data' 'HR' 'Finance' 'data']
# Immediately reveals the typo 'data' vs 'Data'



array(['Data', 'HR', 'Finance', 'data'], dtype=object)

In [5]:
df['City'].unique()
# → ['Mumbai' 'Delhi' 'Pune' 'Bangalore']



array(['Mumbai', 'Delhi', 'Pune', 'Bangalore'], dtype=object)

In [6]:
# Convert to list for cleaner display
df['Dept'].unique().tolist()
# → ['Data', 'HR', 'Finance', 'data']

# WHEN to use unique()?
# First thing you do on any categorical column
# To spot typos, unexpected categories, case inconsistencies
# To understand what values are valid before filtering

['Data', 'HR', 'Finance', 'data']

*unique() immediately exposes data quality issues. The typo 'data' would silently create a fourth group in groupby — your Data department analysis would be wrong and you wouldn't even know it. This is why unique() comes first.*

## Nunique():  how many distint values

In [7]:
# nunique() → returns a SINGLE integer — count of distinct values
# By default excludes NaN from the count

df['Dept'].nunique()       # → 4 (Data, HR, Finance, data)
df['City'].nunique()       # → 4
df['Salary'].nunique()     # → 9 (NaN excluded by default)

# dropna=False → include NaN as a distinct value in count
df['Salary'].nunique(dropna=False)  # → 10

# nunique() on entire DataFrame → count per column
df.nunique()
# Returns a Series: how many unique values each column has

# WHEN to use nunique()?
# Cardinality check — how many categories does this column have?
# High cardinality (many unique values) → not good for groupby
# Low cardinality (few unique values) → good categorical column
# If nunique() == len(df) → every value is unique → likely an ID column

Name      10
Dept       4
City       4
Salary     9
dtype: int64

*df.nunique() on the whole DataFrame is a powerful EDA tool. Name has 10 unique values = every row is unique (ID-like). Dept has 4 = categorical. Salary has 9 = mostly unique numbers. This one line tells you the nature of every column.*

## Value_count(): frequency of each value

In [9]:
# value_counts() → counts occurrences of each unique value
# Returns a Series sorted by frequency DESCENDING by default
# Index = unique values, Values = their counts
# NaN is EXCLUDED by default

df['Dept'].value_counts()



Dept
Data       4
HR         3
Finance    2
data       1
Name: count, dtype: int64

In [10]:
# ascending=True → sort from least to most frequent
df['Dept'].value_counts(ascending=True)



Dept
data       1
Finance    2
HR         3
Data       4
Name: count, dtype: int64

In [11]:
# dropna=False → include NaN count in results
df['Salary'].value_counts(dropna=False)


Salary
60000.0    1
45000.0    1
75000.0    1
55000.0    1
80000.0    1
62000.0    1
71000.0    1
58000.0    1
67000.0    1
NaN        1
Name: count, dtype: int64

In [12]:

# sort=False → keep original category order, don't sort by count
df['City'].value_counts(sort=False)

City
Mumbai       4
Delhi        3
Pune         2
Bangalore    1
Name: count, dtype: int64

*value_counts() instantly reveals the typo 'data' with count 1. Any value with a suspiciously low count (1 or 2) in a categorical column is a red flag — likely a typo or data entry error. Always scan the bottom of value_counts() output.*

## Normalize = True: percentage distribution

In [13]:
# normalize=True → shows PROPORTION instead of raw count
# Values sum to 1.0 (100%)
# Most useful for understanding distribution of categories

df['City'].value_counts(normalize=True)

# Multiply by 100 for percentage
(df['City'].value_counts(normalize=True) * 100).round(1)

# WHEN to use normalize=True?
# "What % of our employees are in Mumbai?"
# "What share of orders are from the Data department?"
# When comparing two datasets of different sizes
# (raw counts are misleading, percentages are fair)

City
Mumbai       40.0
Delhi        30.0
Pune         20.0
Bangalore    10.0
Name: proportion, dtype: float64

*40% of employees are in Mumbai, 30% in Delhi. This is the kind of insight that goes directly into a business report or dashboard slide.*

## Value_counts with bins(): for numeric column

In [14]:
# value_counts() is normally for categorical columns
# But bins parameter makes it work for NUMERIC columns too
# It divides the range into N equal intervals and counts per interval
# This is called binning or bucketing

df['Salary'].value_counts(bins=4)
# Splits Salary range into 4 equal buckets
# Shows how many employees fall in each salary range

# WHY bins?
# Raw salary value_counts gives one row per unique salary
# Not useful — every salary might be unique
# bins groups them into ranges — much more meaningful
# "How many employees earn 40k-55k vs 55k-70k vs 70k-85k?"

# sort=False keeps intervals in ascending order (easier to read)
df['Salary'].value_counts(bins=4, sort=False)

(44964.999, 53750.0]    1
(53750.0, 62500.0]      4
(62500.0, 71250.0]      2
(71250.0, 80000.0]      2
Name: count, dtype: int64

*bins is essentially a histogram in text form. Most employees earn between 55k-75k. This is the kind of quick salary distribution analysis a manager would ask for in 2 minutes.*

## Using value_counts results for filtering and cleaning

In [15]:
# value_counts() returns a Series → you can operate on it

# Get top 2 most frequent cities
top2 = df['City'].value_counts().head(2)
print(top2)

# Get the most frequent value (mode)
most_common = df['City'].value_counts().index[0]
print(most_common)   # → 'Mumbai'

# Filter to keep only cities that appear more than once
city_counts = df['City'].value_counts()
common_cities = city_counts[city_counts > 1].index
df[df['City'].isin(common_cities)]

# Spot and fix typos using value_counts
# value_counts shows 'data' has count 1 → it's a typo
df['Dept'] = df['Dept'].str.title()   # fixes 'data' → 'Data'
df['Dept'].value_counts()              # verify: 'data' gone, Data=5

City
Mumbai    4
Delhi     3
Name: count, dtype: int64
Mumbai


Dept
Data       5
Hr         3
Finance    2
Name: count, dtype: int64

*The typo 'data' (count=1) is now merged into 'Data' (count=5). This is exactly how value_counts drives cleaning decisions — you find the problem, then fix it with str methods.*

## The full EDA checklist using these three methods

In [16]:
# This is the STANDARD first-look routine for any categorical column
# Run this on every text/categorical column before any analysis

for col in ['Dept', 'City']:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"Unique count : {df[col].nunique()}")
    print(f"Unique values: {df[col].unique().tolist()}")
    print(f"Value counts :\n{df[col].value_counts()}")
    print(f"Missing      : {df[col].isnull().sum()}")

# WHY loop over columns?
# In real datasets with 20+ columns you can't check each manually
# This loop gives you a complete profile of every categorical column
# in seconds — you spot ALL data quality issues at once


Column: Dept
Unique count : 3
Unique values: ['Data', 'Hr', 'Finance']
Value counts :
Dept
Data       5
Hr         3
Finance    2
Name: count, dtype: int64
Missing      : 0

Column: City
Unique count : 4
Unique values: ['Mumbai', 'Delhi', 'Pune', 'Bangalore']
Value counts :
City
Mumbai       4
Delhi        3
Pune         2
Bangalore    1
Name: count, dtype: int64
Missing      : 0


*This loop is your standard EDA opening move for categorical columns. Every professional DA runs something like this at the start of every project. In an interview case study, opening with this immediately signals experience.*